<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/01_load_openslr_samples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nepali ASR Transcription Quality Auditor
## Day 1: Load 20 samples from OpenSLR 54

**Goal of the project:** Check if normal ASR scores (WER and CER) are fair for Nepali, by comparing them with what Nepali speakers think.

**Goal of today:** Get 20 audio files with their correct text, and check that they match.

**Runtime:** normal CPU is enough. We do not need a GPU today.

**Data:** OpenSLR 54 (SLR54), license Attribution-ShareAlike 4.0. We use only one zip file (about 590 MB), not the whole dataset.

In [1]:
import csv
from pathlib import Path

import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

In [2]:
work_folder = Path("/content/openslr54")
work_folder.mkdir(exist_ok=True)
print("folder ready:", work_folder)

folder ready: /content/openslr54


In [3]:
base_url = "https://openslr.trmal.net/resources/54"
tsv_url = base_url + "/utt_spk_text.tsv"
zip_url = base_url + "/asr_nepali_0.zip"

!wget -q -c -P {work_folder} {tsv_url}
!wget -q -c -P {work_folder} {zip_url}

In [4]:
# check that both files are there
!ls -lh {work_folder}

total 573M
-rw-r--r-- 1 root root 562M Jun 28  2018 asr_nepali_0.zip
-rw-r--r-- 1 root root  11M Jun 29  2018 utt_spk_text.tsv


In [5]:
!unzip -q -n {work_folder}/asr_nepali_0.zip -d {work_folder}/audio_zip_0

In [6]:
tsv_path = work_folder / "utt_spk_text.tsv"

all_rows = pd.read_csv(
    tsv_path,
    sep="\t",
    header=None,
    names=["file_id", "speaker_id", "text"],
    dtype=str,
    keep_default_na=False,
    quoting=csv.QUOTE_NONE,   # do not treat quote marks in the text in a special way
)

print("rows in tsv:", len(all_rows))
all_rows.head()

rows in tsv: 157905


,file_id,speaker_id,text
0,4aa1fdca33,6a6d1,००७ मिलको दूरीमा
1,86e521554a,1f680,००७ मिलको दूरीमा
2,c96b2b1afb,e349f,००७ मिलको दूरीमा
3,2f7ab41953,294d8,०११ देखि काङ्ग्रेसको
4,c290e5124b,88901,०१ सौर द्रव्यमान


In [7]:
audio_files = list((work_folder / "audio_zip_0").rglob("*.flac"))
print("flac files found:", len(audio_files))
print(audio_files[:3])

# small lookup: file name without .flac -> full path
path_by_id = {}
for file_path in audio_files:
    path_by_id[file_path.stem] = file_path

flac files found: 9935
[PosixPath('/content/openslr54/audio_zip_0/asr_nepali/data/0f/0f1a9a3c10.flac'), PosixPath('/content/openslr54/audio_zip_0/asr_nepali/data/0f/0fff59a959.flac'), PosixPath('/content/openslr54/audio_zip_0/asr_nepali/data/0f/0f36a2c8f7.flac')]


In [8]:
clean_ids = all_rows["file_id"].str.replace(".flac", "", regex=False)
all_rows["audio_path"] = clean_ids.map(path_by_id)

rows_with_audio = all_rows.dropna(subset=["audio_path"])
print("rows with audio:", len(rows_with_audio))

if len(rows_with_audio) == 0:
    print("No match. Compare the file ids in the tsv with the flac names above.")

rows with audio: 9935


In [9]:
sample_rows = rows_with_audio.sample(n=20, random_state=42).reset_index(drop=True)
sample_rows["audio_path"] = sample_rows["audio_path"].astype(str)

sample_rows.to_csv(work_folder / "samples_20.csv", index=False)
sample_rows[["file_id", "text"]].head()

,file_id,text
0,0f5934e285,समेत पाइन्छ
1,0a4ae83ed0,विश्वको अर्थतन्त्रलाई असर
2,0a0f56342e,एमएस्सीका लागि
3,0eb1d8345a,जिरालाई पानीमा पिनेर
4,08f8ed5eeb,तर विडम्बना के छ


In [10]:
sample_rows["seconds"] = [round(sf.info(p).duration, 1) for p in sample_rows["audio_path"]]

print("empty text:", (sample_rows["text"].str.strip() == "").sum())
print("audio missing:", sum(not Path(p).exists() for p in sample_rows["audio_path"]))
print(sample_rows["seconds"].describe())

empty text: 0
audio missing: 0
count    20.000000
mean      3.350000
std       0.819178
min       2.100000
25%       2.775000
50%       3.250000
75%       3.900000
max       5.300000
Name: seconds, dtype: float64


In [11]:
for i in range(3):
    print(i, sample_rows.loc[i, "text"])
    display(Audio(sample_rows.loc[i, "audio_path"]))

0 समेत पाइन्छ


1 विश्वको अर्थतन्त्रलाई असर


2 एमएस्सीका लागि


In [ ]:
!git add .
